## 反馈：

前期的步骤比较完整。在编码之前加入了文件导入和最简单的数据预处理，这一步在项目实战中是必不可少的。现在的txt文件应该仅包含了自我介绍部分，列名称和其他信息仍为手动添加，如果将所有二维表信息都放在excel文件中，一次性用pandas读取，就能够直接生成想要的dataframe结构。
逻辑也比较清楚，先进行one-hot编码，然后计算TF-IDF进行加权，最后在形成文本的向量表示矩阵。在编码之后用dataframe和tfidf_matrix.toarray()对向量矩阵进行的可视化，可以清晰的看到表示结果。

有一点我没太看明白，现在编码基于的词汇表是什么，最后的数据为什么只有五维。一般来讲我们的目标是分析两个文本之间的相似度。此处侧重对自我介绍进行分析。先对所有人的自我介绍进行分词，然后构建词汇表，再对每个句子进行编码，最后计算自我介绍文本之间的相似度，这样就可以找到最相似的两个人。

接下来一周一个是明确一下上面提到的步骤。然后去详细了解以下TfidfVectorizer函数，这个函数其实已经包含了OneHotEncoder函数进行one-hot编码的这一步，可直接进行编码并生成TF-IDF加权后的词袋矩阵。使用.get_feature_names_out()指令能直接创建对应的词汇表。在生成每个句子的词袋模型矩阵后进行相似度计算（可以多用几种计算方法），并且尝试将结果可视化，例如生成热力图。

word2vec可以先不看，除了代码实践外主要了解一下几种相似度计算的方法（余弦相似度、欧氏距离等）的计算公式。

In [22]:
import pandas as pd
import numpy as np
import os
import jieba
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

其中后面四行是scikit-learn库中用于数据预处理的类，
LabelEncoder是标签编码器，将分类文本标签转换为数字，每个类别映射为一个整数
OneHotEncoder是独热编码器，将分类特征转换为独热编码，为每个类别创建一个新的二进制列
ColumnTransformer是列转换器，对数据框的不同列应用不同的预处理转换
CountVectorizer是scikit-learn中用于文本特征提取的工具，它将文本文档集合转换为词频矩阵
TfidfVectorizer是scikit-learn中用于文本特征提取的高级工具，它将文本文档转换为 TF-IDF 特征矩阵

原始数据

In [23]:
text_contents = []
base_path = r'D:\Python study\text'

for i in range(1, 6):
    filename = os.path.join(base_path, f'text_{i}.txt')
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            content = file.read()
            text_contents.append(content)
    except FileNotFoundError:
        print(f"警告: 文件 {filename} 未找到，使用默认内容")
        text_contents.append(f'默认文本内容{i}')
    except Exception as e:
        print(f"读取文件 {filename} 时出错: {e}")
        text_contents.append(f'默认文本内容{i}')

data = pd.DataFrame({
    #分类列
    '城市': ['北京', '贵阳', '杭州', '宁波', '杭州'],
    '性别': ['男', '男', '女', '女', '男'],
    '高考选科': ['物化地', '物化生', '物化政', '物化地', '物化生'],
    #数值列
    '出生月份': [11, 1, 12, 3, 3],
    '生活费': [2000, 1700, 1880, 1560, 2100],
    #文本列
    '自我介绍': text_contents
})
print("原始数据:")
print(data)

原始数据:
   城市 性别 高考选科  出生月份   生活费                                               自我介绍
0  北京  男  物化地    11  2000  大家好，我是来自北京的小王。作为十一月深秋出生的天蝎座，我的性格里既有沉淀的静气，也有不灭的...
1  贵阳  男  物化生     1  1700  朋友们都叫我小李，来自“森林之城”贵阳。作为一月寒冬里出生的摩羯座，我深知“踏实”与“坚韧”...
2  杭州  女  物化政    12  1880  我是小张，一个在西湖边长大的杭州姑娘。十二月出生的射手座，赋予了我乐观自由的灵魂，总渴望探寻...
3  宁波  女  物化地     3  1560  嗨，我是来自海港城市宁波的小陈。身为三月早春出生的双鱼座，我骨子里充满了浪漫的幻想，但港口的...
4  杭州  男  物化生     3  2100  我是小刘，又一个在杭州求学的学子。和三月的春光一样，我的性格活泼而充满生机，对未知永远怀有一...


## 一、OneHotEncoder的使用

### 语法整理
- OneHotEncoder（）
是 scikit-learn 中 OneHotEncoder 类的构造函数语法，用于创建独热编码器实例
语法
```
OneHotEncoder(
    sparse_output=False,  # 输出稠密矩阵而非稀疏矩阵
    drop='first',         # 删除第一个类别以避免多重共线性
    categories='auto',    # 自动检测类别（默认）
    handle_unknown='error' # 处理未知类别的方式
)
```

```·fit_transform（）```

```fit```：学习数据的编码规则（有哪些类别等）

```transform```：应用学到的规则进行实际编码（测试时，应用训练时的模型）

```fit_transform```：一次性完成学习和应用(训练时）

```·get_feature_names_out（）```:获取转换后特征的新名称

```

pd.DataFrame()
pd.DataFrame(
    data=None,      # 数据源
    index=None,     # 行索引
    columns=None,   # 列名
    dtype=None,     # 数据类型
    copy=False      # 是否复制数据
)
```

方法1: 基本的OneHotEncoder使用

In [24]:
categorical_columns = ['城市', '性别', '高考选科']
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_array = encoder.fit_transform(data[categorical_columns])
feature_names = encoder.get_feature_names_out(categorical_columns)
encoded_df = pd.DataFrame(encoded_array,columns=feature_names)
print(encoded_df)

   城市_宁波  城市_杭州  城市_贵阳  性别_男  高考选科_物化政  高考选科_物化生
0    0.0    0.0    0.0   1.0       0.0       0.0
1    0.0    0.0    1.0   1.0       0.0       1.0
2    0.0    1.0    0.0   0.0       1.0       0.0
3    1.0    0.0    0.0   0.0       0.0       0.0
4    0.0    1.0    0.0   1.0       0.0       1.0


方法2: 使用ColumnTransformer处理混合数据类型

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', ['出生月份', '生活费']),  # 数值列直接通过
        ('cat', OneHotEncoder(drop='first'), categorical_columns),  # 分类列进行编码
        ('text', TfidfVectorizer(max_features=10), '自我介绍')  # 文本列使用TF-IDF处理
    ]
)

try:
    # 应用变换
    transformed_data = preprocessor.fit_transform(data)
    print("转换后的数据形状:", transformed_data.shape)

    # 获取所有特征名称
    feature_names = (
        ['出生月份', '生活费'] + 
        list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_columns)) +
        list(preprocessor.named_transformers_['text'].get_feature_names_out())
    )

    # 创建DataFrame（处理可能的稀疏矩阵）
    if hasattr(transformed_data, 'toarray'):
        # 如果是稀疏矩阵，转换为密集数组
        transformed_df = pd.DataFrame(transformed_data.toarray(), columns=feature_names)
    else:
        # 如果已经是密集数组，直接使用
        transformed_df = pd.DataFrame(transformed_data, columns=feature_names)
    
    print("转换后的数据:")
    print(transformed_df)
    
except Exception as e:
    print(f"数据转换过程中出现错误: {e}")

转换后的数据形状: (5, 18)
转换后的数据:
   出生月份     生活费  城市_宁波  城市_杭州  城市_贵阳  性别_男  高考选科_物化政  高考选科_物化生  一个在西湖边长大的杭州姑娘  \
0  11.0  2000.0    0.0    0.0    0.0   1.0       0.0       0.0        0.00000   
1   1.0  1700.0    0.0    0.0    1.0   1.0       0.0       1.0        0.00000   
2  12.0  1880.0    0.0    1.0    0.0   0.0       1.0       0.0        0.57735   
3   3.0  1560.0    1.0    0.0    0.0   0.0       0.0       0.0        0.00000   
4   3.0  2100.0    0.0    1.0    0.0   1.0       0.0       1.0        0.00000   

    一步步向前延伸  一直在我心中熠熠生辉  一部分则投入到我所热爱的志愿者活动中  一部分用于购买人文社科书籍     也坚持健身  \
0  0.000000    0.000000             0.00000        0.00000  0.000000   
1  0.568014    0.568014             0.00000        0.00000  0.000000   
2  0.000000    0.000000             0.57735        0.57735  0.000000   
3  0.000000    0.000000             0.00000        0.00000  0.000000   
4  0.000000    0.000000             0.00000        0.00000  0.690159   

   也有不灭的探索欲       物化地       物化生        选择  
0  0.77828